# Assignment 2 Workbook


Please read and sign the **Academic Integrity Declaration** in myUni Assignment 2 by putting you name and ID. Your assignment may receive 0 mark if this declaration is not signed.



## Question 1a: SVM Primal Form

Please implement the training and testing algorithms of soft-margin Linear Support Vector Machine in its primal form, that is,

$$\min_{\mathbf{w},b,\{\xi_i\}} \frac{1}{2} \|\mathbf{w}\|_2^2 + C\sum_{i=1}^N \xi_i \nonumber \\ s.t.~~ y_i (\mathbf{w}^\top \mathbf{x}_i + b) \ge 1 - \xi_i, ~~\forall i \nonumber \\ \xi_i \ge 0 \nonumber$$
Use CVXPY in your implementation strictly following the format given in this Notebook, and supplying missing code **in the indicated space. Do not add or change any other code**


In [89]:
# import required libraries
!pip install cvxpy --upgrade 
import cvxpy as cp
import pandas as pd
import numpy as np
print("cp", cp.__version__)
print("np",np.__version__)

cp 1.7.1
np 1.26.4


In [90]:
# get training dataset
train = "train1.csv"
df = pd.read_csv(train, header=None)
X_train = df[:1500].iloc[:, 1:].to_numpy()
Y_train = df[:1500].iloc[:, 0].replace(0, -1).to_numpy()

In [91]:
# get test dataset
test = "test1.csv"
df = pd.read_csv(test, header=None)
X_test = df.iloc[:500, 1:].to_numpy()
Y_test = df.iloc[:500, 0].replace(0, -1).to_numpy()

In [92]:
# train linear svm in primal form
def svm_train_primal(data_train, label_train, C):
    X, Y = data_train, label_train
    n_samples, m_features = np.shape(X)
    
    W_value = 0
    b_value = 0
    slack_var_value = 0

# ====================== YOUR CODE HERE ======================  
# The knowledge I used to do this exercise is from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 11, 12, 13, and 14.
# Firstly, we proceed to determine the optimization variables from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 11.
# w: weights vector of shape (m_features,)
# b: bias (intercept), a scalar
# ei: slack variables ξᵢ for each training point (size = number of samples) (n_samples)
# From there, we have the following code:
    w = cp.Variable(m_features)
    b = cp.Variable()
    ei = cp.Variable(n_samples)
# Secondly, from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 11, we continue with defining the constraints. 
# We move on to defining the constraints. From the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 14, we have the following formula:    
# For all i: y_i (w^T x_i + b) >= 1 - ξᵢ
# From there, we have the following code:
    constraints_needing = [Y[i] * (X[i] @ w + b) >= 1 - ei[i] for i in range(n_samples)]
#  Next, we proceed for all slack variables to be >= 0 (ξᵢ ≥ 0), because that is the condition of soft-margin SVM and slack must be non-negative mentioned in the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 14   
    constraints_needing += [ei >= 0]
# Thirdly, we continue to define the objective function, the knowledge used is from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 14
# We have the following formula: min: 0.5 * ||w||^2 + C * sum(ξᵢ)
# From there, we have the following code:
    objective_needing = cp.Minimize(0.5 * cp.norm(w, 2)**2 + C * cp.sum(ei))
# Fourthly, from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 14, we proceed to formulate and solve the optimization problem.
    problem_finding_result = cp.Problem(objective_needing, constraints_needing)
    problem_finding_result.solve() 
# Fifthly, we proceed to extract the solution values:
    W_value = w.value
    b_value = b.value
    slack_var_value = ei.value 
    
# ================================================================

    return [W_value, b_value, slack_var_value]

# train primal model
C = 1
model_primal = svm_train_primal(X_train, Y_train, C)

# output svm primal form solutions
W = model_primal[0]
b = model_primal[1]
slack = model_primal[2]

print('Sum of W, b and slack:')
print(np.round(np.sum(W)+np.sum(slack)+b,2))


Sum of W, b and slack:
443.12


## Question 1b: SVM Primal Accuracy

Please complete and run this code and copy the result into Assignment 2 Question 1b

In [97]:
# predict accuracy of svm model on test dataset
def svm_predict(data_test, label_test, svm_model):
    
    acc = 0.0
    
# ====================== YOUR CODE HERE ======================  
# The knowledge I used to do this exercise is the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 11.
# Firstly, from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 11, we will proceed to extract the weight and bias vectors from the trained SVM model.
# W: the learned weight vector
# b: the learned bias (intercept)
    W = svm_model[0]
    b = svm_model[1]
# Secondly, we continue to calculate the raw score for each test case.
# From the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 11, we have the decision function: f(x) = w^T x + b
    decision_scores_finding = data_test @ W + b
# Thirdly, we apply the sign function to get the predicted label, from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 11
# We have the following decision rule: ŷ = sign(w^T x + b), We can see that sign(score) = +1 or -1, which are the output labels of the SVM  
    predictions_result_finding = np.sign(decision_scores_finding)
# Fourthly, from the seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 1, we compare the predicted label with the actual label.
# Additionally, we count the number of correct predictions.
# From there, we have the following code
    num_correct_counting = np.sum(predictions_result_finding == label_test)
# Fifthly, we proceed to calculate the accuracy.
# We have the following formula for accuracy: Accuracy = (number of correct predictions) / (total number of test samples)
    acc =  num_correct_counting / len(label_test)


# ==========================================================

    return acc

# output primal accuracy as real number, not percentage
accuracy = svm_predict(X_test, Y_test, model_primal)
print('Accuracy:')
print(np.round(accuracy,2))

Accuracy:
0.83


## Question 2a: SVM Dual Form

Please implement the training and testing algorithms of soft-margin Linear Support Vector Machine in its **dual** form, that is,

$$\max_{\alpha_i}\sum_i \alpha_i - \frac{1}{2}\sum_i \sum_j \alpha_i \alpha_j y_i y_j <\mathbf{x}_i, \mathbf{x}_j> \nonumber \\ s.t. ~~~ 0 \le \alpha_i \le C\nonumber \\ ~~~ \sum_i \alpha_i y_i = 0 \nonumber$$

Use CVXPY in your implementation strictly following the format given in this Notebook, and supplying missing code in the indicated space. **Do not modify any other code**.


In [99]:
# import required libraries
import cvxpy as cp
import pandas as pd
import numpy as np


In [101]:
# get training dataset
train = "train1.csv"
df = pd.read_csv(train, header=None)
X_train = df[:1500].iloc[:, 1:].to_numpy()
Y_train = df[:1500].iloc[:, 0].replace(0, -1).to_numpy()

In [103]:
# get test dataset
test = "test1.csv"
df = pd.read_csv(test, header=None)
X_test = df.iloc[:500, 1:].to_numpy()
Y_test = df.iloc[:500, 0].replace(0, -1).to_numpy()

In [105]:
# train linear svm in dual form
def svm_train_dual(data_train, label_train, C):
    
    alphas = 0
    x, y = data_train, label_train
    n_samples, n_features = np.shape(x)
    

# ====================== YOUR CODE HERE ======================  
# Firstly, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 28 and slides lecture week 6 titled "Linear Classifier and (Linear) Support Vector Machine" page 40, we proceed to create dual variable (Lagrange coefficient) alpha to optimize   
# Based on its properties, we can see that each training sample has a corresponding alpha_i
# From there, we can code accordingly as follows:
    alpha_finding = cp.Variable(n_samples) 
# Secondly, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 29, we proceed to calculate the Gram matrix: G_ij = x_i^T x_j    
# We can see that this matrix contains all pairwise dot products between the training samples.
    G_finding = x @ x.T 
# Thirdly, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 29, we continue to calculate the matrix H = y_i * y_j * G_ij   
# By multiplying each element of the outer product of y with the Gram matrix. We can see that this is the matrix used in the quadratic form of the dual objective
# From here, we have the following code:
    H_finding = np.outer(y, y) * G_finding   
# Fourthly, applying the knowledge from seminar slides lecture week 6 titled "Linear Classifier and (Linear) Support Vector Machine" page 46 and 47, we can observe that we pack H into the CVXPY parameter and declare it to be PSD (positive semi-definite) 
# From this we can see that this is required for cvxpy to recognize the quadratic form expression as valid.
    H_parameter_needing = cp.Parameter((n_samples, n_samples), PSD=True)
    H_parameter_needing.value = H_finding    
# Fifthly, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 26, we define the dual objective function appropriate to our problem as: maximize sum(alpha) - 0.5 * alpha.T * H * alpha
# We can observe that it is a dual form representation of the soft-margin SVM optimization problem.
    objective_finding_soft_dual = cp.Maximize(
        cp.sum(alpha_finding)
        - 0.5 * cp.quad_form(alpha_finding, H_parameter_needing)
    )
# Sixthly, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 26, we continue defining the constraints. 
# we can see that each alpha_i must lie in the interval [0, C] and the equality constraint: sum(alpha_i * y_i) = 0
# From this, we can ensure that the separating hyperplane is valid.
    constraints_finding_soft_dual = [
        alpha_finding >= 0,
        alpha_finding <= C,
        cp.sum(cp.multiply(alpha_finding, y)) == 0
    ]
# Seventhly, applying the knowledge from seminar slides lecture week 6 titled "Linear Classifier and (Linear) Support Vector Machine" page 46, 47 and 49, we proceed to solve the quadratic programming problem.
# We use the.OSQP statement to solve Quadratic Programming (QP) problems in CVXPY.
    problem_finding_result = cp.Problem(objective_finding_soft_dual, constraints_finding_soft_dual)
    problem_finding_result.solve(solver=cp.OSQP)    
# Finally, we proceed to store the result into the variable alphas.   
    #return alpha_finding.value
    alphas = alpha_finding.value

# ===========================================================

    # return svm dual model alphas
    return alphas

# train dual model
c = 1
alphas = svm_train_dual(X_train, Y_train, c)

# output svm dual form solutions
print('Sum of alphas:')
print(np.round(np.sum(alphas),2))


C:\Users\Lenovo\anaconda3\Lib\site-packages\cvxpy\reductions\solvers\solving_chain.py:241: UserWarning: You are solving a parameterized problem that is not DPP. Because the problem is not DPP, subsequent solves will not be faster than the first one. For more information, see the documentation on Disciplined Parametrized Programming, at https://www.cvxpy.org/tutorial/dpp/index.html
  warnings.warn(DPP_ERROR_MSG)


Sum of alphas:
443.7


## Question 2b: Dual model parameters

Complete the code where indicated, run it and copy the result into Assignment 2 Question 2b.

In [107]:
# obtain primal w*, b* from dual solution
def find_model_params_from_dual(data, label, alphas, C):
    
    # this value is used to compare values generated by CVXPYY to zero.
    zero_threshold = 0.0001
    n_samples, n_features = np.shape(data)
    a = alphas
   
    a[np.isclose(a, 0, atol=zero_threshold)] = 0  # zero out nearly zeros
    a[np.isclose(a, C, atol=zero_threshold)] = C  # round the ones that are nearly C


# ====================== YOUR CODE HERE ======================  
# Note: b_dual is a scalar, but maybe different for each a parameter, therefore calculate the mean
# Step 1: To begin with, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 28, we proceed to calculate the weight vector w*.
# We have the following formula: w* = = sum_i (alpha_i * y_i * x_i), we proceed to multiply each data point x_i by its corresponding alpha_i and y_i and then sum over all samples to get the weight vector of the separating hyperplane
    w_dual = np.sum((a * label)[:, None] * data, axis=0)
# Step 2: From seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 28, we proceed to determine the support vectors (0 < alpha_i < C)    
# These are also the data points that lie exactly on the margin
    sv_mask_needing = (a > 0) & (a < C)
# Step 3: we proceed to calculate the deviation term b*, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 28, we have the following formula: b* = y_i - w^T x_i
# So, from seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 28, for this case, we calculate b_i = y_i - x_i @ w for each support vector, then average over all those b_i to get a stable estimate of b
    if np.any(sv_mask_needing):
        b_dual = np.mean(label[sv_mask_needing] - data[sv_mask_needing] @ w_dual)
    else:
# From seminar slides lecture week 5 titled "Support Vector Machines 1 part 2" page 28, We can observe that the redundancy is that if no alpha_i is in (0, C), we use all points with alpha_i > 0 and this case can happen due to rounding
        b_dual = np.mean(label[fallback_mask_needing] - data[fallback_mask_needing] @ w_dual)        
   
# =========================================================================
    return w_dual, b_dual

# output reconstructed w* and b* from svm dual problem
C = 1
model_dual = find_model_params_from_dual(X_train, Y_train, alphas, C)

print('Sum of W and b:')
print(np.round(np.sum(model_dual[0])+model_dual[1],2))

Sum of W and b:
0.34


## Question 3: Choosing SVM kernel and soft margin parameter C

In this question, you will apply training/testing methodology to choose two parameters for `sklearn.SVC`, which is Python library implementing classification SVM. This trainint/testing methodology is typical for Machine Learning. Since this could be the first time for many students applying training/testing methodology, all steps are specified.

Read SVC documentation and examples on how to train and test SVC models. (https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)

Please use the following steps **strictly** in order to apply correct methodology and obtain correct final result.

1. Read the training and test data (already coded in the below template, do not change)
2. Split training data into train and validation (dev) sets (already coded in the below template, do not change)

Your code will start from this point.

3. Create four new svc models and train using x_train with the following parameters: `kernel=k, random_state=42`, where k in ('linear', 'poly', rbf', 'sigmoid'). **Leave the rest of options in the model unchanged**. Test using x_dev, get F1 score for each model. 
4. Choose the kernel which has highest F1 score, call it best_kernel.
5. Create five new svc models and train using x_train with the following parameters: `kernel=best_kernel, C=c, random_state=42`, where c in (0.1, 0.25, 0.5, 1.0, 2.0). **Leave the rest of options in the model unchanged**. Test using x_dev, get F1 score for each model.
6. Choose C from point 5 with highest F1. Call it best_c
7. Create new svc model and train using **x_train_full** with the following parameters: `kernel=best_kernel, C=best_c, random_state=42`. **leave the rest of options in the model unchanged**. Test using **x_test**, get F1 score. 
8. Enter output of this code into Question 3 in myUni.
    
 

In [109]:
'''
    This is template for your implementation of Question 3. 
    Do not add any imports
    Do not change any provided code
    Write your code in indicated space only
'''

# imports
import pandas as pd
import numpy as np
from sklearn.svm import SVC as svc
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# 1. Read the training and test data (already coded in the below template, do not change)
train = "train1.csv"
df = pd.read_csv(train, header=None)
x_train_full = df[:1500].iloc[:, 1:].to_numpy()
y_train_full = df[:1500].iloc[:, 0].replace(0, -1).to_numpy()

test = "test1.csv"
df = pd.read_csv(test, header=None)
x_test = df.iloc[:500, 1:].to_numpy()
y_test = df.iloc[:500, 0].replace(0, -1).to_numpy()

# 2. Split training data into train and validation (devset) sets (already coded in the below template, do not change)
x_train, x_dev, y_train, y_dev = train_test_split(x_train_full, y_train_full, test_size=500, random_state=42)

best_f1 = 0 
best_c = 0

#===================== your code is in this space ====================
# To solve this exercise, I used the knowledge from the SVC documentation and examples on how to train and test SVC models. (https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) tutorial and read more material from Workshop 3 Week 6 SVM and Kernels_solns in the section "Task 4: CSV with kernel"
# 3. Combining knowledge from lecture seminar slide week 6 titled "Optimisation and Kernels" page 25 and instructions from questions in part 3, we proceed to test different types of kernels to determine which one performs best on the validation set.
# The list of kernel types to evaluate based on the assignment instructions is:
kernels_given = ['linear', 'poly', 'rbf', 'sigmoid']
# Next, we create a dictionary that stores the F1 score for each kernel.
f1_scores_kernel_dict = {}
# Next, we iterate through each kernel.
for k in kernels_given:
# Based on the question's instructions part 3, we proceed to create an SVC model with a specified kernel and a fixed random state.
    model_finding = svc(kernel=k, random_state=42)    
# Then, we proceed to train the model using the training subset.
    model_finding.fit(x_train, y_train)
# We continue with the prediction on the validation set (dev) based on the instructions of the question in part 3.
    predictions_dev = model_finding.predict(x_dev)
# Next, we calculate the F1 score for the current kernel
    f1_dev = f1_score(y_dev, predictions_dev)
# Finally, we store the F1 score in the dictionary with the kernel as the key.
    f1_scores_kernel_dict[k] = f1_dev 
# 4. Combining knowledge from lecture seminar slide week 6 titled "Optimisation and Kernels" page 26 and instructions from questions in part 4, we come to the step of choosing the kernel with the highest F1 score.
# We use the max() command in python to find the key (kernel) with the highest value of the F1 score and name it best_kernel.
best_kernel = max( f1_scores_kernel_dict, key= f1_scores_kernel_dict.get)
# 5. Combining knowledge from lecture seminar slide week 6 titled "Optimisation and Kernels" page 25 and instructions from questions in part 5, we proceed to tune the regularization parameter C for the best kernel in this step.
# To begin with, based on the instructions of the question in part 5, we create a list of C values to evaluate.
C_values_eva = [0.1, 0.25, 0.5, 1.0, 2.0]
# Next, we continue to create a dictionary that stores the F1 score for each C value.
f1_scores_c_dict = {}
# Next, we iterate through each C value above.
for c in C_values_eva:
# Then, we continue to create SVC model using best_kernel and current C, this step is based on the instructions of the question in part 5, and from lecture seminar slide week 6 titled "Optimisation and Kernels" page 25
    model_finding = svc(kernel=best_kernel, C=c, random_state=42) 
# Then we train the model on the training subset
    model_finding.fit(x_train, y_train)
# Next, we predict on the validation set
    predictions_dev = model_finding.predict(x_dev)
# Finally, we proceed to calculate the F1 score and store it.
    f1_dev = f1_score(y_dev, predictions_dev)
    f1_scores_c_dict[c] = f1_dev
# 6. Combining knowledge from lecture seminar slide week 6 titled "Optimisation and Kernels" page 25, and instructions from questions in part 6, we proceed to choose the best C value for the highest F1 score.   
best_c = max(f1_scores_c_dict, key=f1_scores_c_dict.get)
# 7. Combining knowledge from lecture seminar slide week 6 titled "Optimisation and Kernels" page 25, instructions from questions in part 7 and examples on how to train and test SVC models. (https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html), we will train the final model using the entire training data and the best parameters.
# First of all, we create SVC with the best kernel and best C.
final_model_finding = svc(kernel=best_kernel, C=best_c, random_state=42)
# Next, based on the instructions of the question in part 7, we proceed to train the model on the entire training dataset (1500 samples).
final_model_finding.fit(x_train_full, y_train_full)
# Then, based on the instructions of the question in part 7, we continue to predict on the test set (500 samples)
final_predictions_set = final_model_finding.predict(x_test)
# Finally, we continue to calculate the final F1 score on the test set
best_f1 = f1_score(y_test, final_predictions_set)


#=====================================================================

print("Best F1 and best C")
print(np.round(best_f1+best_c,3))


Best F1 and best C
1.894


## Question 4a: PCA implementation

1. Perform PCA on the dataset to reduce each sample into a 10-dimensional feature vector.
2. Print the required result and enter into Assignment 2 Question 3a.

=========================================================================
- Implementing PCA algorithm.
    - Start
        - Input: $m$ number of samples as matrix $X$ of $m$ rows and $n$ columns.
        - Calculate the mean vector for each column. $$mean = \frac {1}{m} \sum \limits _{i=1} ^{n}X_{ij}$$
        - Calculate the centralised matrix $X_C$ and covariance matrix $C$. $$X_C=X-mean$$ $$C = \frac {1}{m}(X_C)^TX_C$$
        - Calculate the eigenvalues and eigenvectors using convariance matrix.
        - Select top $k$ principal components - as eigen vector corresponding to top $k$ eigen values. Construct matrix $P$.
    - End
    
- Transforming the the data using the principal components (matrix $P$) obtained using the PCA algorithm. $$Transformed \: Data = X_C P$$
- Calculating the covariance matrix of the transformed data by first centralising it (mean subtracted) and then obtaining the covariance matrix.

In [111]:
# import required libraries
import pandas as pd
import numpy as np

In [113]:
# get training dataset
train = "train1.csv"
df = pd.read_csv(train, header=None)
X = df[:1500].iloc[:, 1:].to_numpy()

In [115]:
# Selecting top 10 Principal components
no_of_components = 10

covariance_matrix_X = 0
covariance_matrix_X_transformed = 0

# ====================== YOUR CODE HERE ======================  
# In this exercise, I use the knowledge from the lecture seminar slide week 7 titled "Dimensionality and complexity reduction" and the instructions of the problem to solve this exercise.
# Step 1: From lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 5, we start with calculating the mean vector of each feature (column)
# We perform this operation which will be used to centralize the data.
mean_vector_needing = np.mean(X, axis=0)  # shape: (n_features,)
# Step 2: From lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 6, We center the data by subtracting the mean from each row.
# From this, we can see that each row in Xc is a centered version of the original data.
X_C_finding = X - mean_vector_needing  # shape: (m_samples, n_features)
# Step 3: From lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 6, We proceed to calculate the covariance matrix of the centralized data.
# From the knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 6, we know that the covariance matrix measures the linear relationship between the features.
m_samples = X.shape[0]  # number of samples
covariance_matrix_X = (X_C_finding.T @ X_C_finding) / m_samples  # shape: (n_features, n_features)
# Step 4: From the knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 21, we proceed to calculate the eigenvalues and eigenvectors of the covariance matrix.
# We can observe that from the knowledge of lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 21, the eigenvectors determine the direction of maximum variance (the components themselves).
# Additionally, we use the np.linalg.eigh() function in Python to compute the eigenvalues and eigenvectors of a real symmetric matrix because the covariance matrix we are using is symmetric.
eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix_X)
# Step 5: From the lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 21 and the instructions from the question, we proceed to arrange the eigenvalues in descending order and take the top-k index, thereby the eigenvectors corresponding to the largest eigenvalues retain the largest variance.
# Besides, we use the np.argsort() function in Python to return the index of the elements sorted in ascending order of an array.
sorted_indices_finding = np.argsort(eigenvalues)[::-1]  # descending order
top_k_indices_needing = sorted_indices_finding[:no_of_components]  # keep top 10 indices
# Step 6: Combining knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 18 and instructions from the question, we proceed to build projection matrix P from k top-k eigenvectors
# We can observe in detail that each column of P is an eigenvector (principal component)
P = eigenvectors[:, top_k_indices_needing]  # shape: (n_features, no_of_components)
# Step 7: Combining knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 18 and instructions from the question, we proceed to transform data using projection matrix P.
# We multiply the centered data by P to reduce the dimensionality.
X_transformed = X_C_finding @ P  # shape: (m_samples, no_of_components)
# Step 8: Combining the knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 5 and the instructions from the question, we proceed to calculate the average value of the transformed data (according to each principal component)
mean_transformed_finding = np.mean(X_transformed, axis=0)  # shape: (no_of_components,)
# Step 9: Combining the knowledge from the lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 6 and the instructions from the question, we proceed to re-center the transformed data (minus the mean value).
X_transformed_C_result = X_transformed - mean_transformed_finding  # shape: (m_samples, no_of_components)
# Step 10: Combining the knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 6 and the instructions from the question, we proceed to calculate the covariance matrix of the transformed data.
covariance_matrix_X_transformed = (X_transformed_C_result.T @ X_transformed_C_result) / m_samples  # shape: (10, 10)

# ==========================================================================

sum_cov_X = np.sum(covariance_matrix_X)
sum_cov_X_transformed = np.sum(covariance_matrix_X_transformed)

print("sum_cov_X_transformed:")
print(np.round(sum_cov_X + sum_cov_X_transformed,2))

sum_cov_X_transformed:
325.08


## Question 4b: PCA data reconstructon

For this question:
1. Reconstruct the X dataset from your results in code Question 3a as X_back.
1. Centre  X_back and calculate covariance matrix for X_back and enter its sum into Assignment 2 Question 3b.

For this part, use the libraries imported in Question 4a, and do not import any moore libraries.

In [117]:
# Do not import any additional libraries for this section
# Do not change any code outside of this area marked with =============================

# ====================== YOUR CODE HERE ======================  
# The knowledge used to solve this exercise is that I use the knowledge from the lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 20 and the instructions from the question.
# Step 1: Combining the knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 20 and the instructions from the question, we proceed to reconstruct the original dataset from the PCA transformed data.
# We observe that X_transformed has shape (m_samples, no_of_components) and P.T has shape (no_of_components, n_features). Then we multiply them which will give an approximation of the original data in the full feature space.
X_back_finding = X_transformed @ P.T  # shape: (m_samples, n_features)
# Step 2: Combining the knowledge from lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 5 and the instructions from the question, we proceed to calculate the mean value of the reconstructed data. We need the mean value to center the reconstructed data before calculating the covariance.
mean_X_back_final = np.mean(X_back_finding, axis=0)  # shape: (n_features,)
# Step 3: Combining the knowledge from the lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 6 and the instructions from the question, we need to center the reconstructed data by subtracting its mean. This ensures a zero mean for each feature, a requirement for accurate covariance calculation.
X_back_C_result = X_back_finding - mean_X_back_final  # shape: (m_samples, n_features)
# Step 4: Combining the knowledge from the lecture seminar slide week 7 titled "Dimensionality and complexity reduction" page 6 and the instructions from the question, we calculate the covariance matrix of the centralized reconstruction data. 
# We have the same formula as in PCA: (X^T X) / m
m_samples_finding = X_back_finding.shape[0]  # number of samples, same as original data
covariance_matrix_X_back = (X_back_C_result.T @ X_back_C_result) /m_samples_finding  # shape: (n_features, n_features)

# ============================================================
print("Sum of covariance_matrix_X_back:")
print(np.round(np.sum(covariance_matrix_X_back),2))

Sum of covariance_matrix_X_back:
68.06


## Questions 5a: Kernel k-Means derivation

In this question you will derive a new mathematical formula for Kernel k=Means and then implement it.

The task is specified as follows:
1. We have a dataset with M instances and N features in each instance. We can express this dataset as a set of n-dimensional vectors $\mathbf{X}=\{\mathbf{x}_i \}\in \mathbb{R}^{M\times N}$. Each of these vectors is $\mathbf{x}_i \in \mathbb{R}^N$. 

2. We have a vector of averages $\mathbf{\mu} = \frac{1}{M}\sum_i \mathbf{x}_i$, where i-th value of that vector is an average of i-th feature over the entire $X$ dataset. Therefore $\mu$ is an "average instance" for that dataset and $\mathbf{\mu} \in \mathbb{R}^N$.

3. Let $m_1 = \frac{1}{M^2}\sum_i\sum_j \|\mathbf{x}_i - \mathbf{x}_j\|_2^2$ which is average of squared pairwise distance between instances. 

4. Let $m_2 = \frac{1}{M} \sum_i\|\mathbf{x}_i - \mathbf{\mu}\|_2^2$ which is variance of distances between each $\mathbf{x}_i$ and mean $\mu$.

Note that $m_1$ and $m_2$ are scalars.

5. k-Means with RBF-kernel is given as follows:

$$k(\mathbf{x}_i,\mathbf{x}_j) = \exp(\frac{-\|\mathbf{x}_i - \mathbf{x}_j\|_2^2}{2\sigma^2})$$

As you can see $2\sigma^2$ can be expressed in terms of $m_1$, which gives you

$$k(\mathbf{x}_i,\mathbf{x}_j) = \exp(\frac{-\|\mathbf{x}_i - \mathbf{x}_j\|_2^2}{2m_2})$$

Your tasks are as follows:
1. Derive the relationship between $m_1$ and $m_2$. Let $m_2=f(m_1)$ be the dependency between $m_1$ and $m_2$.

Important: **Include all steps and transformations that you use and comment on them by answering the "why" question (why this equals that).** Points may be deducted if a transformation is not clearly explained.

2. Replace $2m_2$ with your derived $f(m_1)$ in the formula of k-Means with RBF-kernel


$$k(\mathbf{x}_i,\mathbf{x}_j) = \exp(\frac{-\|\mathbf{x}_i - \mathbf{x}_j\|_2^2}{f(m_1)})$$


You can use derivation space below or use derivation space in myUni

===============================================================================




## Question 5b: Kernel k-Means implemention


In Question 5a you have proved the relationship between $m_1$ and $m_2$ as $m_2=f(m_1)$

In this question you will implement kernel k-means with modified RBF-kernel, using $m_2=f(m_1)$, and using SpectralClustering from sklearn to reduce the programming efford of the full implementation. 
RBF-kernel to implement is as follows:

$$k(\mathbf{x}_i,\mathbf{x}_j) = \exp(\frac{-\|\mathbf{x}_i - \mathbf{x}_j\|_2^2}{f(m_1)})$$


In order to calculate paiwise distance matrix, please use `scipy.spatial` distance function `cdist`, which is alread pre-loaded in the code. <br>
Please check the documentation of the SpectralClustering function to use precomputed affinity matrix, which you are going to supply with the above specification.


**NOTE:** to calculate distance, use `distance.cdist` with `"sqeuclidean"`


For reference, please write **below** the full formula of your RBF kernel replaceing $f(m_1)$ with your derivation 


In [121]:
import pandas as pd
import numpy as np
from sklearn.cluster import SpectralClustering
from scipy.spatial import distance

df = pd.read_csv('train1.csv', header = None)
X = df.iloc[:1000,1:].to_numpy(copy=True)
Y = df.iloc[:1000,:1].to_numpy(copy=True)

In [123]:
n_clusters = 10
centroids_rbf = np.array(n_clusters)

centroids_rbf = []

# ====================== YOUR CODE HERE ======================  
"""
# use random_state=0 in the SpectralClustering function to make the results reproducible.
"""
# To solve this exercise, I used the knowledge from the results of the proof of question 5a, the instructions of the question, from the lecture seminar slide week 8 titled "Unsupervised Learning part 2" and from the lecture seminar slide week 1 titled "Introduction to Statistical Machine Learning"
# Step 1: Combining the instructions from question 5a and from the lecture seminar slide week 1 titled "Introduction to Statistical Machine Learning" page 47, we start calculating the squared Euclidean distance for each pair.
# We see that the instructions in question 5a , give us the formula to calculate ||x_i - x_j||^2 for all i, j. Through the instructions in question 5B , we apply scipy.spatial.distance.cdist to calculate the full distance matrix. Also, most importantly, 'sqeuclidean' is guaranteed to return the squared distance (not the normal distance).
# From the data from the problem above, I can know that the output shape: (1000, 1000)
pairwise_sq_distsance_needing = distance.cdist(X, X, "sqeuclidean")
# Step 2: Combine the instructions from question 5a above on how to calculate m1 and the knowledge from lecture seminar slide week 8 titled "Unsupervised Learning part 2" page 23, and 24. I proceed to calculate m1 by averaging the square of the distance between each pair.
# Based on the instructions from question 5a above on how to calculate m1, we have the following formula: m1 = (1 / M^2) * sum_{i,j} ||x_i - x_j||^2, where M = number of data points
# From there, we code accordingly as follows:
# Firstly, we will start with M equal to the number of data points
M = X.shape[0]  # M = 1000
# Then, we proceed to code as follows: # np.sum(pairwise_sq_distsance_needing) sums over all i,j to calculate m1
# I will further explain why m1 is calculated in my opinion. In question 5a above, we deduced that the RBF kernel becomes: k(x_i, x_j) = exp( - ||x_i - x_j||^2 / f(m1 )). From this, we can see that this helps avoid choosing an arbitrary sigma and makes it data-driven.
m1 = np.sum(pairwise_sq_distsance_needing) / (M * M)
# Step 3: Combining the instructions from question 5a about the formula k(x_i, x_j) = exp( - ||x_i - x_j||^2 / f(m1 )) and from the lecture seminar slide week 8 titled "Unsupervised Learning part 2" page 24, we proceed to calculate the RBF Multiplication Matrix using m1.
# We can see that for each i, j in the distance matrix, we perform the calculation K[i, j] = exp( - D[i, j] / m1 ). From my perspective, we observe that this calculation converts distance into similarity: large distance → small similarity.
# Here we will implement the code to calculate the Kernel matrix, the result will be symmetric and positive semi-definite
kernel_matrix_result = np.exp(-pairwise_sq_distsance_needing / m1)
# Step 4: Combining the instructions from question 5b and from the lecture seminar slide week 8 titled "Unsupervised Learning part 2" page 24, we continue with applying Spectral Clustering using multiplication matrices.
# Following the instructions of question 5b, we can see that SpectralClustering expects an affinity matrix when using precomputed mode. Additionally, we follow the instructions of question 5b that random_state=0 ensures consistent clustering results in each run, since spectral clustering uses randomness in the eigenvector/k-means selection step.
# From there, we have the following code:
model_needing = SpectralClustering(
    n_clusters=n_clusters,
    affinity='precomputed',  # means we are passing our own kernel matrix
    random_state=0
)
# I created this additional code to run the Spectral Clustering model with the kernel matrix I calculated above.
cluster_labels_need = model_needing.fit_predict(kernel_matrix_result)  # <== this we create cluster_labels

# Step 6: Combining the instructions of question 5b and from the lecture seminar slide week 8 titled "Unsupervised Learning part 2" page 23, and 24, we proceed to calculate the centroid for each cluster (in the original feature space).
# Following the guidance of question 5b, we observe that we want to compute the “centroid” of each cluster, that is, the mean of all the samples assigned to that cluster in the input space.
# From the lecture seminar slide week 8 titled "Unsupervised Learning part 2" page 8, and 24. We will start by iterating over all the cluster indices.
for k in range(n_clusters):
# Then, we will proceed to get the points in X belonging to cluster k, thereby filtering the rows in X based on the cluster label   
    cluster_points_finding = X[cluster_labels_need == k]
# Next, from the lecture seminar slide week 8 titled "Unsupervised Learning part 2" page 8, and 24, we will observe if the cluster has members, let's calculate the cluster centroid (average of rows)
    if len(cluster_points_finding) > 0:
        centroid_result = np.mean(cluster_points_finding, axis=0)  # shape: (N_features,)
# Additionally, we can see that if no points are assigned to this cluster (which is likely), use the zero vector
    else:
        centroid_result = np.zeros(X.shape[1])
# Next, we will add the calculated centroids to the list.
    centroids_rbf.append(centroid_result)
# Next, we convert the list of centroids into a NumPy array and define the shape: (n_clusters, N_features)
centroids_rbf = np.array(centroids_rbf)

# =================================================================
    
print('Sum of centroids:')
print(np.round(np.sum(centroids_rbf),2))


C:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(


Sum of centroids:
36.78


## Question 6a: Adaboost questions

Answer questions in myUni assignment 2

## Question 6b: Adaboost code correction
The following code shows an incomplete implementation of Adaboost algorithm. The code does not implement the full logic specific to Adaboost. Please modify it to make a correct implementation. After completing the implementation, run the code and copy the result into space provided in myUni Question 6.

1. **Do not delete anything, just add what is required to existing code. Adding code is sufficient to make correct implementation** <br>
2. **You can add code only in sections as indicated.** <br>
3. **Please clearly comment the purpose of the code that you added.**
4. **Do not change or add any imports**



In [125]:
# Do not change this section

# import required libraries
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from math import exp, log as ln

# DO NOT use any other import statements for this question

In [127]:
# Do not change this section

# get training dataset
train = "train1.csv"
df = pd.read_csv(train, header=None)
X_train = df.iloc[:, 1:].to_numpy()
Y_train = df.iloc[:, 0].replace(0, -1).to_numpy()

In [129]:
# Do not change this section

# get test dataset
test = "test1.csv"
df = pd.read_csv(test, header=None)
X_test = df.iloc[:, 1:].to_numpy()
Y_test = df.iloc[:, 0].replace(0, -1).to_numpy()

In [131]:
'''
You can add code in this section, without deleteing anything
DO NOT add any imports
Comment anything that you addeded
'''

# To solve this exercise I used knowledge from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" pages 24 and 25.
# To begin with, , from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we start by adding a global list to store the alpha_t for each weak learner.
# We then store αₜ (the weight of each weak classifier). This value will be used later in the experiment.
alphas_initial = []

# sample_weights for training data
# Then, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we define a function to train a weak classifier, we add sample weights, and now we have 3 parameters: training data, labels, and sample weights.
def weak_classifier_train(train_data, train_label, sample_weight_needing):   # Added the sample weight argument
    # Create a decision stump
    stump = DecisionTreeClassifier(max_depth=1)

# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we train the decision model using weighted samples, the 'sample_weight' parameter ensures that the model pays more attention, and
    # Train the stump using the weighted samples
    stump.fit(train_data, train_label, sample_weight=sample_weight_needing)    # pass the sample weights into fit()
    
    return stump

def weak_classifier_predict(test_data, model):
    
    return model.predict([test_data])


def Adaboost_train(train_data, train_label, T):

# train_data: N x d matrix
# train_label: N x 1 vector
# T: the number of weak classifiers in the ensemble
   
# Then, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we initialize the number of training examples
    m_finding = len(train_data)
# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24 , we initialize the uniform weights D₀(i) = 1/m, where D is the weight distribution
# I use the np.ones() function in python in this part to create an array with all elements equal to 1. 
    D_result = np.ones(m_finding) / m_finding
# We observe that the problem has given us a set of weak learners by the name ensemble_models       
    ensemble_models = [] 
    for t in range(0,T):
        # model_param_t returns the model parameters of the learned weak classifier
# From the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24 and 25, we proceed to train the weak classifier hₜ using the current weights Dₜ
        model = weak_classifier_train(train_data, train_label, D_result)    # Train using D_result
# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we observe that above we have trained the weak learner with weights Dₜ, so we will make predictions on the training set
        predictions_needing = model.predict(train_data)   # Predict labels on training data using hₜ
# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we proceed to calculate the weighted error εₜ = ∑ Dₜ(i) * 𝟙[hₜ(xᵢ) ≠ yᵢ]        
        incorrect_finding = (predictions_needing != train_label)   # Boolean Array: We see that true when the prediction is wrong
        epsilon_result = np.sum(D_result * incorrect_finding)   # Weighted error rate εₜ
# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we avoid invalid values so we force epsilon to (0, 0.5]
        if epsilon_result == 0:
            epsilon_result = 1e-10   # Avoid log(0)
        elif epsilon_result > 0.5:        
# Next, we use the "continue" code to skip the poor classifiers.
             continue  
# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we calculate alphaₜ using the formula: alphaₜ = 0.5 * ln((1 - εₜ) / εₜ)
        alpha_t_needing = 0.5 * ln((1 - epsilon_result) / epsilon_result)   
# Next, we store the above alphaₜ to predict the test time.
        alphas_initial.append(alpha_t_needing)      # Store αₜ globally
# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we update Dₜ₊₁(i) = Dₜ(i) * exp(-αₜ yᵢ hₜ(xᵢ)) / Z and apply the exponential update        
        D_result = D_result * np.exp(-alpha_t_needing * train_label * predictions_needing)       
# Next, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" page 24, we normalize the weights so that their sum is 1 (Zₜ in the slide).       
        D_result = D_result / np.sum(D_result)     # Normalize so that ∑ Dₜ₊₁(i) = 1   
           
        ensemble_models.append(model)
    
    return ensemble_models

def Adaboost_test(test_data, ensemble_models):
# test_data: n x d
    predictions = []

    for i in range(len(test_data)):
        decision_ensemble = 0
        
        for k in range(len(ensemble_models)):
# Then, from the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" pages 24 and 25, we proceed to predict the label hₖ(xᵢ) for the current test sample using the weak model k.
# We see that weak classifier prediction usually returns a list, so we use [0] to extract the scalar result.
            prediction = weak_classifier_predict(test_data[i], ensemble_models[k])[0]  # hₖ(xᵢ) ∈ {−1, +1}
# From the lecture seminar slide week 9 titled "Decision Trees and Ensembles part 2 of 2" pages 24 and 25, we multiply the prediction by the corresponding weight alphaₖ, where alphaₖ reflects the confidence level placed on the weak classifier k. Then, we continue to add the weighted prediction to the sum H(xᵢ), we can see that it will gradually increase the cumulative vote: H(xᵢ) ← H(xᵢ) + αₖ hₖ(xᵢ)
            decision_ensemble += alphas_initial[k] * prediction    # Weighted vote: αₖ * hₖ(xᵢ)
            
            if decision_ensemble > 0:
                prediction = 1
            else:
                prediction = -1
        predictions.append(prediction)
            
    return predictions

# predict and output accuracy

ensemble_models = Adaboost_train(X_train, Y_train, 3)
predicted_labels = Adaboost_test(X_test, ensemble_models)

In [133]:
######## do not change this code ############

# Calculate accuracy
accuracy = accuracy_score(Y_test, predicted_labels)

print('Accuracy:')
print(np.round(accuracy,3))

Accuracy:
0.876
